In [0]:

# AlertaBairro - Camada Prata
# Etapa 1: Expandir os registros da Bronze

from pyspark.sql import functions as F

bronze_alagamentos = spark.table(
    "workspace.alertabairro.bronze_alagamentos"
)

bronze_clima = spark.table(
    "workspace.alertabairro.bronze_clima"
)

# Transformar cada ocorrencia em uma linha
ocorrencias = bronze_alagamentos.select(
    F.explode("features").alias("ocorrencia")
)

# Transformar cada data em uma linha
dias = bronze_clima.select(
    F.explode("daily.time").alias("data")
)

print("=== ALAGAMENTOS ===")
print("Total de entradas:", ocorrencias.count())
print(
    "IDs distintos:",
    ocorrencias.select(
        "ocorrencia.properties.cd_identificador"
    ).distinct().count()
)

print("=== CLIMA ===")
print("Total de entradas:", dias.count())
print("Datas distintas:", dias.distinct().count())


=== ALAGAMENTOS ===
Total de entradas: 584
IDs distintos: 292
=== CLIMA ===
Total de entradas: 16
Datas distintas: 10


In [0]:

# AlertaBairro - Camada Prata
# Etapa 2: Estruturacao e tipagem dos dados

from pyspark.sql import functions as F

# CLIMA: separar datas e valores de precipitacao
df_clima_prata = (
    bronze_clima
    .select(
        F.posexplode("daily.time").alias("indice", "data_texto"),
        F.col("daily.precipitation_sum").alias("chuvas")
    )
    .select(
        F.to_date("data_texto").alias("data"),
        F.col("chuvas")[F.col("indice")].cast("double").alias("precipitacao_mm")
    )
)

# ALAGAMENTOS: extrair os campos das ocorrencias
df_alagamentos_prata = (
    bronze_alagamentos
    .select(F.explode("features").alias("f"))
    .select(
        F.col("f.properties.cd_identificador").alias("id_ocorrencia"),
        F.to_date(
            F.substring("f.properties.dt_ocorrencia", 1, 10)
        ).alias("data_ocorrencia"),
        F.col("f.properties.dc_tipo_ocorrencia").alias("tipo_ocorrencia"),
        F.col("f.properties.nm_distrito").alias("distrito"),
        F.col("f.properties.nm_subprefeitura").alias("subprefeitura"),
        F.col("f.geometry.coordinates")[0].cast("double").alias("longitude"),
        F.col("f.geometry.coordinates")[1].cast("double").alias("latitude"),
        F.to_date(
            F.substring("f.properties.dt_carga", 1, 10)
        ).alias("data_carga_origem")
    )
)

print("=== CLIMA ESTRUTURADO ===")
df_clima_prata.show(5, truncate=False)

print("=== ALAGAMENTOS ESTRUTURADOS ===")
df_alagamentos_prata.show(5, truncate=False)

print("Total clima:", df_clima_prata.count())
print("Total alagamentos:", df_alagamentos_prata.count())


=== CLIMA ESTRUTURADO ===
+----------+---------------+
|data      |precipitacao_mm|
+----------+---------------+
|2026-10-28|1.6            |
|2026-10-29|20.2           |
|2026-10-30|5.2            |
|2026-10-31|1.7            |
|2026-11-01|17.8           |
+----------+---------------+
only showing top 5 rows
=== ALAGAMENTOS ESTRUTURADOS ===
+-------------+---------------+---------------+--------+----------------------------+-----------+------------+-----------------+
|id_ocorrencia|data_ocorrencia|tipo_ocorrencia|distrito|subprefeitura               |longitude  |latitude    |data_carga_origem|
+-------------+---------------+---------------+--------+----------------------------+-----------+------------+-----------------+
|1458         |2026-04-01     |ALAGAMENTO     |NULL    |VP - VILA PRUDENTE          |-46.5368208|-23.60621642|2026-10-06       |
|1472         |2026-04-02     |ALAGAMENTO     |NULL    |VP - VILA PRUDENTE          |-46.5301669|-23.61379432|2026-10-06       |
|1494      

In [0]:

# AlertaBairro - Camada Prata
# Etapa 3: Validacao de qualidade

from pyspark.sql import functions as F

print("=== QUALIDADE DOS DADOS CLIMATICOS ===")

df_clima_prata.select(
    F.count("*").alias("total_registros"),
    F.countDistinct("data").alias("datas_unicas"),
    F.sum(
        F.col("data").isNull().cast("int")
    ).alias("datas_nulas"),
    F.sum(
        F.col("precipitacao_mm").isNull().cast("int")
    ).alias("precipitacoes_nulas"),
    F.sum(
        (F.col("precipitacao_mm") < 0).cast("int")
    ).alias("precipitacoes_negativas")
).show()

print("=== QUALIDADE DOS ALAGAMENTOS ===")

df_alagamentos_prata.select(
    F.count("*").alias("total_registros"),
    F.countDistinct("id_ocorrencia").alias("ids_unicos"),
    F.sum(
        F.col("id_ocorrencia").isNull().cast("int")
    ).alias("ids_nulos"),
    F.sum(
        F.col("data_ocorrencia").isNull().cast("int")
    ).alias("datas_nulas"),
    F.sum(
        F.col("distrito").isNull().cast("int")
    ).alias("distritos_nulos"),
    F.sum(
        F.col("subprefeitura").isNull().cast("int")
    ).alias("subprefeituras_nulas"),
    F.sum(
        (
            F.col("latitude").isNull() |
            F.col("longitude").isNull()
        ).cast("int")
    ).alias("coordenadas_nulas")
).show()


=== QUALIDADE DOS DADOS CLIMATICOS ===
+---------------+------------+-----------+-------------------+-----------------------+
|total_registros|datas_unicas|datas_nulas|precipitacoes_nulas|precipitacoes_negativas|
+---------------+------------+-----------+-------------------+-----------------------+
|             16|          10|          0|                  0|                      0|
+---------------+------------+-----------+-------------------+-----------------------+

=== QUALIDADE DOS ALAGAMENTOS ===
+---------------+----------+---------+-----------+---------------+--------------------+-----------------+
|total_registros|ids_unicos|ids_nulos|datas_nulas|distritos_nulos|subprefeituras_nulas|coordenadas_nulas|
+---------------+----------+---------+-----------+---------------+--------------------+-----------------+
|            584|       292|        0|          0|            584|                   0|                0|
+---------------+----------+---------+-----------+---------------+-

In [0]:

# AlertaBairro - Camada Prata
# Etapa 4: Deduplicacao dos registros

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# === CLIMA ===

# Identificar o periodo final de cada lote
clima_lotes = (
    bronze_clima
    .select(
        F.array_max("daily.time").alias("ultima_data_lote"),
        F.posexplode("daily.time").alias("indice", "data_texto"),
        F.col("daily.precipitation_sum").alias("chuvas")
    )
    .select(
        F.to_date("data_texto").alias("data"),
        F.col("chuvas")[F.col("indice")].cast("double").alias("precipitacao_mm"),
        F.to_date("ultima_data_lote").alias("ultima_data_lote")
    )
)

janela_clima = Window.partitionBy("data").orderBy(
    F.col("ultima_data_lote").desc(),
    F.col("precipitacao_mm").desc_nulls_last()
)

df_clima_limpo = (
    clima_lotes
    .withColumn("rn", F.row_number().over(janela_clima))
    .filter(F.col("rn") == 1)
    .drop("rn", "ultima_data_lote")
)

# === ALAGAMENTOS ===

# Desempate estavel quando a data de carga e igual

# === REGRAS DE QUALIDADE ===

condicao_valida = (
    F.col("id_ocorrencia").isNotNull()
    & (F.length(F.trim(F.col("id_ocorrencia").cast("string"))) > 0)
    & F.col("data_ocorrencia").isNotNull()
    & F.col("subprefeitura").isNotNull()
    & (F.length(F.trim(F.col("subprefeitura"))) > 0)
    & F.col("latitude").between(-90, 90)
    & F.col("longitude").between(-180, 180)
)

df_alagamentos_validos = df_alagamentos_prata.filter(
    condicao_valida
)

print("=== VALIDACAO DE QUALIDADE ===")
print("Registros recebidos:", df_alagamentos_prata.count())
print("Registros validos:", df_alagamentos_validos.count())
print(
    "Registros barrados:",
    df_alagamentos_prata.count() - df_alagamentos_validos.count()
)

colunas = [F.col(c) for c in df_alagamentos_prata.columns]

alagamentos_com_hash = df_alagamentos_validos.withColumn(
    "_hash",
    F.sha2(F.to_json(F.struct(*colunas)), 256)
)

janela_alagamentos = Window.partitionBy("id_ocorrencia").orderBy(
    F.col("data_carga_origem").desc_nulls_last(),
    F.col("_hash").desc()
)

df_alagamentos_limpo = (
    alagamentos_com_hash
    .withColumn("rn", F.row_number().over(janela_alagamentos))
    .filter(F.col("rn") == 1)
    .drop("rn", "_hash")
)

# === VALIDACAO ===

print("=== RESULTADO DA DEDUPLICACAO ===")

print("Clima antes:", df_clima_prata.count())
print("Clima depois:", df_clima_limpo.count())

print("Alagamentos antes:", df_alagamentos_prata.count())
print("Alagamentos depois:", df_alagamentos_limpo.count())

print("=== CLIMA CONSOLIDADO ===")
df_clima_limpo.orderBy("data").show(15)


=== VALIDACAO DE QUALIDADE ===
Registros recebidos: 584
Registros validos: 584
Registros barrados: 0
=== RESULTADO DA DEDUPLICACAO ===
Clima antes: 16
Clima depois: 10
Alagamentos antes: 584
Alagamentos depois: 292
=== CLIMA CONSOLIDADO ===
+----------+---------------+
|      data|precipitacao_mm|
+----------+---------------+
|2026-10-28|            1.6|
|2026-10-29|           20.2|
|2026-10-30|            5.2|
|2026-10-31|            1.7|
|2026-11-01|           17.8|
|2026-11-02|            6.9|
|2026-11-03|            0.0|
|2026-11-04|            2.3|
|2026-11-05|            3.1|
|2026-11-06|            8.3|
+----------+---------------+



In [0]:

# AlertaBairro - Camada Prata
# Etapa 5: Criar tabelas Delta vazias

tabela_clima = "workspace.alertabairro.prata_clima"
tabela_alagamentos = "workspace.alertabairro.prata_alagamentos"

# Criar tabelas com os schemas dos dados tratados
# limit(0) preserva as colunas, sem inserir registros
# mode("ignore") evita recriar tabelas existentes

(
    df_clima_limpo.limit(0)
    .write.format("delta")
    .mode("ignore")
    .saveAsTable(tabela_clima)
)

(
    df_alagamentos_limpo.limit(0)
    .write.format("delta")
    .mode("ignore")
    .saveAsTable(tabela_alagamentos)
)

print("=== TABELAS PRATA ===")

for nome in [tabela_clima, tabela_alagamentos]:
    print("Tabela:", nome)
    print("Registros:", spark.table(nome).count())
    print("----------------------------")

print("Tabelas criadas/verificadas com sucesso!")


=== TABELAS PRATA ===
Tabela: workspace.alertabairro.prata_clima
Registros: 0
----------------------------
Tabela: workspace.alertabairro.prata_alagamentos
Registros: 0
----------------------------
Tabelas criadas/verificadas com sucesso!


In [0]:

# AlertaBairro - Camada Prata
# Etapa 6: MERGE dos dados climaticos

from delta.tables import DeltaTable

# Abrir a tabela Delta de destino
delta_clima = DeltaTable.forName(
    spark,
    "workspace.alertabairro.prata_clima"
)

# Inserir novos dias ou atualizar valores existentes
(
    delta_clima.alias("destino")
    .merge(
        df_clima_limpo.alias("origem"),
        "destino.data = origem.data"
    )
    .whenMatchedUpdateAll(
        condition="""
            NOT (
                destino.precipitacao_mm
                <=> origem.precipitacao_mm
            )
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("=== MERGE CLIMA CONCLUIDO ===")

spark.sql("""
    SELECT
        COUNT(*) AS total_registros,
        COUNT(DISTINCT data) AS datas_unicas,
        MIN(data) AS primeira_data,
        MAX(data) AS ultima_data
    FROM workspace.alertabairro.prata_clima
""").show()


=== MERGE CLIMA CONCLUIDO ===
+---------------+------------+-------------+-----------+
|total_registros|datas_unicas|primeira_data|ultima_data|
+---------------+------------+-------------+-----------+
|             10|          10|   2026-10-28| 2026-11-06|
+---------------+------------+-------------+-----------+



In [0]:

# AlertaBairro - Camada Prata
# Etapa 7: MERGE dos alagamentos

from delta.tables import DeltaTable

delta_alagamentos = DeltaTable.forName(
    spark,
    "workspace.alertabairro.prata_alagamentos"
)

# Colunas que podem sofrer atualizacoes
colunas_comparacao = [
    "data_ocorrencia",
    "tipo_ocorrencia",
    "distrito",
    "subprefeitura",
    "longitude",
    "latitude",
    "data_carga_origem"
]

# Comparacao segura, inclusive para valores NULL
condicao_atualizacao = " OR ".join(
    f"NOT (destino.{col} <=> origem.{col})"
    for col in colunas_comparacao
)

# Executar MERGE por identificador unico
(
    delta_alagamentos.alias("destino")
    .merge(
        df_alagamentos_limpo.alias("origem"),
        "destino.id_ocorrencia = origem.id_ocorrencia"
    )
    .whenMatchedUpdateAll(
        condition=condicao_atualizacao
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("=== MERGE ALAGAMENTOS CONCLUIDO ===")

spark.sql("""
    SELECT
        COUNT(*) AS total_registros,
        COUNT(DISTINCT id_ocorrencia) AS ids_unicos,
        MIN(data_ocorrencia) AS primeira_ocorrencia,
        MAX(data_ocorrencia) AS ultima_ocorrencia
    FROM workspace.alertabairro.prata_alagamentos
""").show()


=== MERGE ALAGAMENTOS CONCLUIDO ===
+---------------+----------+-------------------+-----------------+
|total_registros|ids_unicos|primeira_ocorrencia|ultima_ocorrencia|
+---------------+----------+-------------------+-----------------+
|            292|       292|         2026-01-04|       2026-09-28|
+---------------+----------+-------------------+-----------------+



In [0]:

# AlertaBairro - Etapa 8
# Verificar o ultimo MERGE

historico = spark.sql("""
    DESCRIBE HISTORY
    workspace.alertabairro.prata_alagamentos
""")

historico.select(
    "version",
    "operation",
    "operationMetrics"
).show(3, truncate=False)


+-------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|operation|operationMetrics                                                                                                                                                                                                                                                                                      

In [0]:

# AlertaBairro - Camada Prata
# Etapa 9: Auditoria das duas execucoes MERGE

from pyspark.sql import functions as F

historico = spark.sql("""
    DESCRIBE HISTORY
    workspace.alertabairro.prata_alagamentos
""")

metricas = (
    historico
    .filter(F.col("operation") == "MERGE")
    .orderBy(F.col("version").desc())
    .limit(2)
)

metricas.select(
    "version",
    "operation",
    F.col("operationMetrics")["numTargetRowsInserted"].alias("inseridos"),
    F.col("operationMetrics")["numTargetRowsUpdated"].alias("atualizados"),
    F.col("operationMetrics")["numTargetRowsDeleted"].alias("excluidos")
).show(truncate=False)


+-------+---------+---------+-----------+---------+
|version|operation|inseridos|atualizados|excluidos|
+-------+---------+---------+-----------+---------+
|3      |MERGE    |0        |0          |0        |
|2      |MERGE    |0        |0          |0        |
+-------+---------+---------+-----------+---------+



In [0]:

# AlertaBairro - Camada Prata
# Etapa 10: Auditoria do MERGE climatico

from pyspark.sql import functions as F

historico_clima = spark.sql("""
    DESCRIBE HISTORY
    workspace.alertabairro.prata_clima
""")

metricas_clima = (
    historico_clima
    .filter(F.col("operation") == "MERGE")
    .orderBy(F.col("version").desc())
    .limit(2)
)

metricas_clima.select(
    "version",
    "operation",
    F.col("operationMetrics")["numTargetRowsInserted"].alias("inseridos"),
    F.col("operationMetrics")["numTargetRowsUpdated"].alias("atualizados"),
    F.col("operationMetrics")["numTargetRowsDeleted"].alias("excluidos")
).show(truncate=False)


+-------+---------+---------+-----------+---------+
|version|operation|inseridos|atualizados|excluidos|
+-------+---------+---------+-----------+---------+
|2      |MERGE    |0        |0          |0        |
|1      |MERGE    |10       |0          |0        |
+-------+---------+---------+-----------+---------+



In [0]:

%sql
SELECT
    to_json(features[0].properties) AS propriedades
FROM workspace.alertabairro.bronze_alagamentos
LIMIT 1;


propriedades
"{""cd_identificador"":""1458"",""dc_tipo_ocorrencia"":""ALAGAMENTO"",""dt_carga"":""2026-10-06Z"",""dt_ocorrencia"":""2026-04-01Z"",""nm_subprefeitura"":""VP - VILA PRUDENTE"",""sg_fonte_original"":""SIGRC""}"


In [0]:

%sql
SELECT
    f.properties.nm_distrito AS distrito_origem,
    f.properties.nm_subprefeitura AS subprefeitura_origem
FROM workspace.alertabairro.bronze_alagamentos b
LATERAL VIEW EXPLODE(b.features) e AS f
LIMIT 10;


distrito_origem,subprefeitura_origem
null,VP - VILA PRUDENTE
null,VP - VILA PRUDENTE
null,BT - BUTANTA
null,AF - ARICANDUVA/VILA FORMOSA
null,AF - ARICANDUVA/VILA FORMOSA
null,BT - BUTANTA
null,BT - BUTANTA
null,CL - CAMPO LIMPO
null,CV - CASA VERDE/CACHOEIRINHA
null,LA - LAPA


In [0]:

%sql
WITH origem AS (
    SELECT
        TRIM(f.properties.cd_identificador)
            AS id_ocorrencia,
        TRY_CAST(
            SUBSTRING(f.properties.dt_ocorrencia, 1, 10)
            AS DATE
        ) AS data_ocorrencia,
        TRIM(f.properties.nm_subprefeitura)
            AS subprefeitura,
        TRY_CAST(f.geometry.coordinates[0] AS DOUBLE)
            AS longitude,
        TRY_CAST(f.geometry.coordinates[1] AS DOUBLE)
            AS latitude
    FROM workspace.alertabairro.bronze_alagamentos b
    LATERAL VIEW EXPLODE(b.features) e AS f
)
SELECT
    COUNT(*) AS total_registros,
    COUNT_IF(
        id_ocorrencia IS NULL OR id_ocorrencia = ''
    ) AS rejeitados_id,
    COUNT_IF(
        data_ocorrencia IS NULL
    ) AS rejeitados_data,
    COUNT_IF(
        subprefeitura IS NULL OR subprefeitura = ''
    ) AS rejeitados_regiao,
    COUNT_IF(
        latitude IS NULL OR longitude IS NULL
        OR latitude NOT BETWEEN -90 AND 90
        OR longitude NOT BETWEEN -180 AND 180
    ) AS rejeitados_coordenadas,
    COUNT(*) - COUNT(DISTINCT id_ocorrencia)
        AS registros_repetidos_por_id
FROM origem;


total_registros,rejeitados_id,rejeitados_data,rejeitados_regiao,rejeitados_coordenadas,registros_repetidos_por_id
584,0,0,0,0,292
